<a href="https://colab.research.google.com/github/Tanmay-Somani/100daysof-code/blob/main/Main_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install torch torchvision torchaudio
!pip install opencv-python tqdm matplotlib scikit-learn

In [4]:
import os
import glob
import re
import logging
from datetime import datetime
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random

In [27]:
SEQ_LEN = 8
PRED_STEPS = 1
IMG_SIZE = 1500
BATCH_SIZE = 4
EPOCHS = 40
LR = 5e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED=42



In [28]:
BASE_DIR = "/content/project"

os.makedirs(BASE_DIR, exist_ok=True)

DATA_ROOT = f"{BASE_DIR}/data"
LOG_PATH = f"{BASE_DIR}/logs"
LOG_FILE=LOG_PATH+ "/train_cropped.log"
MODEL_PATH = f"{BASE_DIR}/models"
CHKPOINT_DIR = f"{MODEL_PATH}/checkpoints"
for p in [DATA_ROOT, LOG_PATH, MODEL_PATH, CHKPOINT_DIR]:
    os.makedirs(p, exist_ok=True)

In [29]:
with open(LOG_PATH+'/runno.txt','r') as  R:
    runno=int(R.read())
with open(LOG_PATH+'/runno.txt','w') as  W:
    W.write(str(int(runno+1)))
logging.basicConfig(level=logging.INFO,format="%(asctime)s | %(levelname)s | %(message)s",
                    handlers=[logging.FileHandler(LOG_FILE),
                              logging.StreamHandler()]
)

In [30]:

def set_seed(seed):
    """
    Sets the seeds for `random`, `numpy`, and `torch` libraries to ensure reproducibility.

    Args:
        seed (int): The seed value to be used.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [31]:




def parse_datetime(path):
    """
    Parses a datetime string from a file path.

    The file path is expected to be in the format "MMWYY_HHMM" or "MWWYY_HHMM", where:
        - MM: Month as a two-digit number (01-12)
        - W: Weekday (JAN-FEB etc.)
        - YY: Year as a four-digit number
        - HH: Hour in 24-hour format (00-23)
        - MM: Minute (00-59)

    Args:
        path (str): The file path to parse the datetime from.

    Returns:
        datetime: A datetime object representing the parsed date and time.

    Raises:
        ValueError: If the file path does not match the expected format.
    """
    name = os.path.basename(path)
    pattern = r"(\d{2})(\w{3})(\d{4})_(\d{4})"
    match = re.search(pattern, name)
    if not match:
        raise ValueError(f"Bad filename: {name}")
    month_map = {
        'JAN': '01','FEB': '02','MAR': '03','APR': '04','MAY': '05','JUN': '06',
        'JUL': '07','AUG': '08','SEP': '09','OCT': '10','NOV': '11','DEC': '12'
    }
    day=match.group(1)
    month = match.group(2)
    year=match.group(3)
    time=match.group(4)
    for key, value in month_map.items():
        if key in month:
            month = value
            break
    return datetime.strptime(f"{day}{month}{year}.{time}", "%d%m%Y.%H%M")


In [32]:


def mse(p,t):
    """
    Computes the mean squared error (MSE) between two tensors.

    Args:
        p (torch.tensor): The predicted tensor.
        t (torch.tensor): The target tensor.

    Returns:
        torch.tensor: The mean squared error value.
    """
    return torch.mean((p-t)**2)

def rmse(p,t):
    """
    Computes the root mean squared error (RMSE) between two tensors.

    Args:
        p (torch.tensor): The predicted tensor.
        t (torch.tensor): The target tensor.

    Returns:
        torch.tensor: The root mean squared error value.
    """
    return torch.sqrt(mse(p,t)+1e-8)

def mae(p,t):
    """
    Computes the mean absolute error (MAE) between two tensors.

    Args:
        p (torch.tensor): The predicted tensor.
        t (torch.tensor): The target tensor.

    Returns:
        torch.tensor: The mean absolute error value.
    """
    return torch.mean(torch.abs(p-t))

def encode_time(dt):
    """
    Encodes a datetime object into sinusoidal representation.

    The encoding is based on the hour of the day, with values scaled to be between -1 and 1.

    Args:
        dt (datetime): The datetime object to encode.

    Returns:
        tuple: A pair of numpy arrays representing the sine and cosine components of the encoded time.
    """
    hour = dt.hour + dt.minute / 60.0
    sin = np.sin(2 * np.pi * hour / 24)
    cos = np.cos(2 * np.pi * hour / 24)
    return sin, cos

def loss_fn(pred, target):
    mse = ((pred-target)**2).mean()
    mae = torch.abs(pred-target).mean()
    return mse + 0.1*mae, mse.item(), mae.item()



In [33]:
class SpatioTemporalDataset(Dataset):
    """
    Initializes the SpatioTemporalDataset class.

    Args:
        root (str): The root directory of the dataset.

    Note:
        This function loads all PNG images in the specified directory and its subdirectories,
        sorts them by datetime, and creates a list of tuples containing input and target sequences.

    Attributes:
        samples (list[tuple]): A list of tuples, where each tuple contains an input sequence and a corresponding target sequence.
    """
    def __init__(self, root):
        all_imgs = glob.glob(os.path.join(root, "**/*.png"), recursive=True)
        all_imgs = sorted(all_imgs, key=parse_datetime)
        self.samples = []
        total_len = SEQ_LEN + PRED_STEPS


        for i in range(len(all_imgs) - total_len):
            window = all_imgs[i:i+total_len]
            inp = window[:SEQ_LEN]
            tgt = window[SEQ_LEN:]
            self.samples.append((inp, tgt))

        logging.info(f"Total valid samples: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def load_img(self, path):
        """
        Load an image from a file.

        Args:
            path (str): The path to the image file.

        Returns:
            np.ndarray: The loaded image.
        """
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = img / 255.0
        return img

    def __getitem__(self, idx):
        seq = []
        targets = []
        inp_paths, tgt_paths = self.samples[idx]

        for p in inp_paths:
            img = self.load_img(p)
            dt = parse_datetime(p)
            sin, cos = encode_time(dt)
            time_map = np.ones_like(img)
            stacked = np.stack([
            img,
            sin * time_map,
            cos * time_map
            ], axis=0)
            seq.append(stacked)
        seq = np.stack(seq, axis=0)

        for p in tgt_paths:
            img = self.load_img(p)
            targets.append(np.expand_dims(img, axis=0))
            targets = np.stack(targets, axis=0)
            return torch.tensor(seq, dtype=torch.float32), torch.tensor(targets, dtype=torch.float32)



In [34]:
class ConvLSTMCell(nn.Module):
    """
    A 2D Convolutional LSTM Cell.

    Attributes:
        hidden_dim (int): The number of output features in the cell.
        conv (nn.Conv2d): The convolutional layer that combines input and hidden state.

    Methods:
        forward(x, h, c): The forward pass through the cell.
    """
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.conv = nn.Conv2d(
        in_dim + hidden_dim,
        4 * hidden_dim,
        3,
        padding=1
        )

    def forward(self, x, h, c):
        """
        The forward pass through the cell.

        Args:
            x (torch.Tensor): The input tensor.
            h (torch.Tensor): The current hidden state.
            c (torch.Tensor): The current cell state.

        Returns:
            tuple: A tuple containing the updated hidden and cell states.
        """

        combined = torch.cat([x, h], dim=1)
        out = self.conv(combined)
        i, f, o, g = torch.chunk(out, 4, dim=1)
        i=torch.sigmoid(i)
        f=torch.sigmoid(f)
        o=torch.sigmoid(o)
        g=torch.tanh(g)
        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c

# =========================
# MODEL
# =========================
class ConvLSTMModel(nn.Module):
    """
    A 2D Convolutional LSTM Model.

    Attributes:
        cell1 (ConvLSTMCell): The first convolutional LSTM cell.
        cell2 (ConvLSTMCell): The second convolutional LSTM cell.
        head (nn.Conv2d): The final convolutional layer that produces the output.

    Methods:
        forward(x): The forward pass through the model.
    """
    def __init__(self):
        super().__init__()
        self.cell1 = ConvLSTMCell(3, 16)
        self.cell2 = ConvLSTMCell(16, 16)
        self.head = nn.Conv2d(16, PRED_STEPS, kernel_size=1)

    def forward(self, x):
        """
        The forward pass through the model.

        Args:
            x (torch.Tensor): The input tensor.

        Returns:
            torch.Tensor: The output tensor.
        """

        B, T, C, H, W = x.shape
        h1 = torch.zeros(B, 16, H, W).to(x.device)
        c1 = torch.zeros(B, 16, H, W).to(x.device)
        h2 = torch.zeros(B, 16, H, W).to(x.device)
        c2 = torch.zeros(B, 16, H, W).to(x.device)
        for t in range(T):
            h1, c1 = self.cell1(x[:, t], h1, c1)
            h2, c2 = self.cell2(h1, h2, c2)
        out = self.head(h2) # (B, PRED_STEPS, H, W)
        return out.unsqueeze(2) # (B, T_pred, C=1, H, W)

def save_predictions(pred, target, epoch):
    """
    Save predictions to files.

    Args:
        pred (torch.Tensor): The predicted tensor.
        target (torch.Tensor): The target tensor.
        epoch (int): The current epoch.
    """

    pred = pred.detach().cpu().numpy()
    target = target.detach().cpu().numpy()
    for i in range(min(2, pred.shape[0])):
        for t in range(PRED_STEPS):
            plt.imsave(f"/home/trainee/Desktop/Project_1/images/preds/preds_main_cropped/epoch{epoch}_sample{i}_t{t}.png",pred[i, t, 0], cmap='gray')

def save_ckpt(model, opt, epoch):
    """
    Save a checkpoint to file.

    Args:
        model (nn.Module): The model to be saved.
        opt (torch.optim.Optimizer): The optimizer to be saved.
        epoch (int): The current epoch.
    """
    os.makedirs(CHKPOINT_DIR, exist_ok=True)
    path = os.path.join(CHKPOINT_DIR, f"epoch_{epoch}.pth")
    torch.save({
    "model": model.state_dict(),
    "opt": opt.state_dict(),
    "epoch": epoch
    }, path)
    logging.info(f"Checkpoint saved: {CHKPOINT_DIR}")



In [35]:
def train():
    dataset = SpatioTemporalDataset(DATA_ROOT)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,num_workers=2)
    logging.info("dataset loaded")
    model = ConvLSTMModel().to(DEVICE)
    logging.info("Model Architecture")
    logging.info(model)
    hist = {"loss":[],"mae":[],"mse":[],"rmse":[]}
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)

    try:
        start_epoch = int(input("Enter the epoch number to resume training from (or 0 for full training): "))
    except ValueError:
        logging.warning("Invalid input. Resuming training from epoch 0.")
        start_epoch = 0

    if start_epoch > 0 and os.path.exists(CHKPOINT_DIR):
        # Load checkpoint
        path = os.path.join(CHKPOINT_DIR, f"epoch_{start_epoch-1}.pth")
        checkpoint = torch.load(path)
        model.load_state_dict(checkpoint["model"])
        opt.load_state_dict(checkpoint["opt"])
        logging.info(f"Resuming training from epoch {start_epoch}...")
    else:
        start_epoch = 0

    for epoch in range(start_epoch,EPOCHS):
        totals = {k:0 for k in hist}
        cnt=0
        model.train()
        total_loss, total_mse, total_mae = 0,0,0
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}")

        for x, y in pbar:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            pred = model(x)
            loss, mse_val, mae_val = loss_fn(pred,y)
            loss.backward()
            opt.step()

            with torch.no_grad():
                totals["loss"] += loss.item()
                totals["mae"] += mae(pred,y).item()
                totals["mse"] += mse(pred,y).item()
                totals["rmse"] += rmse(pred,y).item()
                cnt += 1

            metrics = {k:totals[k]/cnt for k in totals}
            for k in hist: hist[k].append(metrics[k])
            pbar.set_postfix(loss=loss.item())

        avg = total_loss / len(loader)
        logging.info(f"Epoch {epoch+1} ")
        n = len(loader)
        torch.save(model.state_dict(), MODEL_PATH+"/model_CNNLSTM.pth")
        save_ckpt(model, opt, epoch)
        save_predictions(pred, y, epoch)

        if epoch % 5 ==0 :
            torch.save(model.state_dict(), MODEL_PATH+f"/model_CNNLSTM_1.{epoch}.pth")

        logging.info("Plotting Now")

        for k,v in hist.items():
            plt.figure()
            plt.plot(v)
            plt.ylabel(k)
            plt.xlabel("steps")
            plt.title(k)
            plt.grid()
            plt.savefig(f"/home/trainee/Desktop/Project_1/images/plots/plots_main_cropped/{k}.png")
            plt.close()

    logging.info("Training complete")



In [36]:
if __name__ == "__main__":
    logging.info('----------------------------------------------------------------')
    logging.info("Starting Project...")
    logging.info(f"Run No. {runno}")
    logging.info("Configurations: ")
    logging.info("Sequence length of images taken "+ str(SEQ_LEN))
    logging.info("Current Batch Size taken " + str(BATCH_SIZE))
    logging.info("Number of Epochs in the run "+str(EPOCHS))
    logging.info("Current training image resolution " + str(IMG_SIZE))
    logging.info("Learning Rate "+str(LR))
    logging.info(f"DEVICE {DEVICE}")
    set_seed(SEED)
    try:
        train()
    except Exception as e:
        logging.exception(f"Fatal error: {e}")

Enter the epoch number to resume training from (or 0 for full training): 0


Epoch 1: 0it [00:00, ?it/s]
ERROR:root:Fatal error: division by zero
Traceback (most recent call last):
  File "/tmp/ipykernel_8175/373830148.py", line 14, in <cell line: 0>
    train()
  File "/tmp/ipykernel_8175/3466563722.py", line 53, in train
    avg = total_loss / len(loader)
          ~~~~~~~~~~~^~~~~~~~~~~~~
ZeroDivisionError: division by zero
